# HOPE: prune and evaluate OLMoE-1B-7B on Kaggle

End-to-end pipeline: **download → fp16 conversion → calibrate → solve → prune → evaluate**, with a formatted comparison at the end.

**Before running**, in the notebook sidebar (*Settings*):
- **Accelerator:** GPU T4 ×2
- **Internet:** on
- *(Optional)* add a Kaggle dataset with your `eval_prompts.txt` (and calibration prompts), then set the paths in the config cell below.

Every heavy step runs as a separate process (`!hope ...`), so GPU/CPU memory is freed between steps. Large files (model copies) go to `/tmp`; small results go to `/kaggle/working` so they are saved with the notebook output.

## 1. Configuration

In [ ]:
REPO_URL = "https://github.com/alfey504/HOPE-adapted.git"
MODEL_ID = "allenai/OLMoE-1B-7B-0924"
DTYPE = "float16"            # T4 has no native bfloat16; switch to "bfloat16" if you see non-finite errors

# Prompts: one per line (.txt) or a JSON list of strings.
# If a file doesn't exist, a fallback is used (WikiText for calibration, demo prompts for eval).
CALIB_PROMPTS = "/kaggle/input/hope-prompts/calib_prompts.txt"
EVAL_PROMPTS = "/kaggle/input/hope-prompts/eval_prompts.txt"
NUM_FALLBACK_CALIB = 256     # number of WikiText paragraphs used if CALIB_PROMPTS is missing

PRUNE_FRAC = 0.25            # fraction of experts to prune per layer (64 -> 48 for 0.25)
BASELINE = "reap"            # first-order baseline to compare against: "reap", "ean", "man", "freq", or None
MAX_PROMPT_LENGTH = 2048     # truncate calibration prompts (tokens)

EVAL_MAX_NEW_TOKENS = 128
EVAL_LIMIT = None            # e.g. 50 for a quick run
NUM_GENERATIONS_TO_SHOW = 5

# Paths
REPO_DIR = "/kaggle/working/HOPE-adapted"
MODEL_DIR = "/tmp/olmoe-" + DTYPE
OUT_DIR = "/kaggle/working/hope-out"
HOPE_MODEL_DIR = "/tmp/pruned-hope"
BASELINE_MODEL_DIR = "/tmp/pruned-" + str(BASELINE)
EVAL_DIR = OUT_DIR + "/eval"

import os
os.environ["HF_HOME"] = "/tmp/hf"                 # keep the ~14 GB download off /kaggle/working
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.makedirs(OUT_DIR, exist_ok=True)

## 2. Environment check

In [ ]:
import shutil, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
for path in ["/tmp", "/kaggle/working"]:
    free = shutil.disk_usage(path).free / 1e9
    print("Free disk at %-16s %6.1f GB" % (path, free))
print("Needs roughly: /tmp ~%d GB (model + pruned copies), /kaggle/working < 1 GB"
      % (14 + 11 * (2 if BASELINE else 1)))

## 3. Clone and install

Rerunning this cell pulls the latest code. The notebook kernel never imports `torch`/`transformers` (all heavy steps run as subprocesses), so no kernel restart is needed after installing.

In [ ]:
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
!pip install -q -e . datasets
# Check versions in a fresh process: importing torch/transformers into this kernel right
# after pip changes packages can leave half-loaded modules (e.g. "sympy has no attribute printing")
!python -c "import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__); assert transformers.__version__ == '5.2.0'"

## 4. Prepare prompts

In [ ]:
import json

def read_prompts(path):
    with open(path) as f:
        if path.endswith(".json"):
            return json.load(f)
        return [line.strip() for line in f if line.strip()]

def write_prompts(prompts, path):
    with open(path, "w") as f:
        json.dump(prompts, f)
    return path

if os.path.exists(CALIB_PROMPTS):
    calib_path = CALIB_PROMPTS
else:
    import datasets
    print("CALIB_PROMPTS not found; using %d WikiText-2 paragraphs" % NUM_FALLBACK_CALIB)
    wiki = datasets.load_dataset("wikitext", "wikitext-2-raw-v1", split="train")["text"]
    paras = [t.strip() for t in wiki if len(t.strip()) > 300 and not t.strip().startswith("=")]
    calib_path = write_prompts(paras[:NUM_FALLBACK_CALIB], OUT_DIR + "/calib_prompts.json")

if os.path.exists(EVAL_PROMPTS):
    eval_path = EVAL_PROMPTS
else:
    print("WARNING: EVAL_PROMPTS not found; using a few demo prompts")
    eval_path = write_prompts([
        "Write a Python function that computes the Fibonacci sequence.",
        "Explain the theory of general relativity in simple terms.",
        "What are the main differences between TCP and UDP?",
        "The capital of France is",
        "Solve the equation 2x + 5 = 17.",
    ], OUT_DIR + "/eval_prompts.json")

print("Calibration prompts: %d (%s)" % (len(read_prompts(calib_path)), calib_path))
print("Eval prompts:        %d (%s)" % (len(read_prompts(eval_path)), eval_path))

## 5. Download and convert the model

In [ ]:
if not os.path.exists(MODEL_DIR + "/config.json"):
    !python scripts/download_model.py --model-id {MODEL_ID} --out-path {MODEL_DIR} --dtype {DTYPE} --sanity-check
    !rm -rf /tmp/hf   # free the download cache
else:
    print("Using existing", MODEL_DIR)

## 6. Calibrate (collect expert interaction statistics)

In [ ]:
obs_path = OUT_DIR + "/observations.h5"
!hope calibrate --model-path {MODEL_DIR} --prompts {calib_path} --out-path {obs_path} --max-prompt-length {MAX_PROMPT_LENGTH}

## 7. Solve for the prune-sets

In [ ]:
hope_pruneset = OUT_DIR + "/pruneset_hope.json"
!hope solve --obs-path {obs_path} --prune-frac {PRUNE_FRAC} --out-path {hope_pruneset}
if BASELINE:
    baseline_pruneset = OUT_DIR + "/pruneset_%s.json" % BASELINE
    !hope baselines --obs-path {obs_path} --prune-frac {PRUNE_FRAC} --method {BASELINE} --out-path {baseline_pruneset}

## 8. Prune

In [ ]:
!hope prune --model-path {MODEL_DIR} --pruneset-path {hope_pruneset} --out-path {HOPE_MODEL_DIR}
if BASELINE:
    !hope prune --model-path {MODEL_DIR} --pruneset-path {baseline_pruneset} --out-path {BASELINE_MODEL_DIR}

## 9. Evaluate against the original

In [ ]:
# The original model's results are cached in EVAL_DIR/original_cache and reused on reruns,
# so only the pruned models are evaluated again. Add --recompute-original to force a fresh run.
pruned_args = "--pruned " + HOPE_MODEL_DIR
if BASELINE:
    pruned_args += " --pruned " + BASELINE_MODEL_DIR
limit_arg = "--limit %d" % EVAL_LIMIT if EVAL_LIMIT else ""
!python scripts/eval_prompts.py --original {MODEL_DIR} {pruned_args} --prompts {eval_path} --out-dir {EVAL_DIR} --max-new-tokens {EVAL_MAX_NEW_TOKENS} {limit_arg}

## 10. Results

In [ ]:
import html
import pandas as pd
from IPython.display import display, HTML, Markdown

summary = json.load(open(EVAL_DIR + "/summary.json"))
df = pd.DataFrame(summary).T
orig_ppl = df.loc["original", "perplexity"]
df["ppl_change"] = df["perplexity"] / orig_ppl - 1
df = df.reindex(columns=["perplexity", "ppl_change", "kl", "top1_agreement",
                         "exact_match", "num_nonfinite"])
df.columns = ["Perplexity", "Δ PPL vs original", "KL (↓)", "Top-1 agreement (↑)",
              "Exact-match generations (↑)", "Non-finite prompts"]
df.index.name = "Model"

num_pruned = len(next(iter(json.load(open(hope_pruneset)).values())))
display(Markdown("### Summary (%d experts pruned per layer, %.0f%%)"
                 % (num_pruned, PRUNE_FRAC * 100)))
display(
    df.style
    .format({
        "Perplexity": "{:.3f}",
        "Δ PPL vs original": "{:+.2%}",
        "KL (↓)": "{:.4f}",
        "Top-1 agreement (↑)": "{:.1%}",
        "Exact-match generations (↑)": "{:.1%}",
        "Non-finite prompts": "{:.0f}",
    }, na_rep="—")
    .highlight_min(subset=["KL (↓)"], color="#d4edda")
    .highlight_max(subset=["Top-1 agreement (↑)", "Exact-match generations (↑)"], color="#d4edda")
)

pruned_rows = df.drop(index="original")
if len(pruned_rows) > 1:
    best = pruned_rows["KL (↓)"].idxmin()
    display(Markdown("**Closest to the original (lowest KL): `%s`**" % best))

In [ ]:
# Side-by-side generations for the first few prompts
rows = [json.loads(line) for line in open(EVAL_DIR + "/generations.jsonl")]
models = [k for k in rows[0] if k != "prompt"]

def fmt_metrics(m):
    parts = []
    if "perplexity" in m:
        parts.append("ppl %.2f" % m["perplexity"])
    if "kl" in m:
        parts.append("KL %.4f" % m["kl"])
    if "top1_agreement" in m:
        parts.append("top-1 %.0f%%" % (100 * m["top1_agreement"]))
    return " · ".join(parts)

style = ("<style>.gen td,.gen th{vertical-align:top;text-align:left;padding:6px;"
         "border:1px solid #ddd}.gen pre{white-space:pre-wrap;margin:0;font-size:12px}"
         ".gen small{color:#888}</style>")
for i, row in enumerate(rows[:NUM_GENERATIONS_TO_SHOW]):
    cells_html = "".join(
        "<td><b>%s</b><br><small>%s</small><pre>%s</pre></td>"
        % (html.escape(name), fmt_metrics(row[name]), html.escape(row[name]["generation"]))
        for name in models
    )
    display(HTML(style + "<p><b>Prompt %d:</b> %s</p><table class='gen'><tr>%s</tr></table>"
                 % (i + 1, html.escape(row["prompt"]), cells_html)))

## 11. Saved outputs

In [ ]:
for f in sorted(os.listdir(OUT_DIR)) + ["eval/" + f for f in sorted(os.listdir(EVAL_DIR))]:
    path = os.path.join(OUT_DIR, f)
    if os.path.isfile(path):
        print("%-40s %8.1f KB" % (f, os.path.getsize(path) / 1e3))
print("\nPruned models (not saved with notebook output):", HOPE_MODEL_DIR,
      BASELINE_MODEL_DIR if BASELINE else "")
# To keep the HOPE-pruned model (~11 GB, fits in the 20 GB /kaggle/working limit), uncomment:
# !cp -r {HOPE_MODEL_DIR} /kaggle/working/olmoe-pruned-hope